# Домашнее задание 4 по теме 6

**Дисциплина** Язык Python для разработчиков

**Тема** Аннотации типов и валидация данных

**Цель задания** Научиться пользоваться аннотацией типов и технологиями для валидации данных

# Задача «Система управления библиотекой»

В этом задании вы создадите систему управления библиотекой, используя аннотации типов Python и Pydantic.

## Задача 1. Базовые модели

Создайте следующие Pydantic-модели:

**Book**
- `title`: str
- `author`: str
- `year`: int
- `available`: bool

**User**
- `name`: str
- `email`: str (с валидацией email)
- `membership_id`: str

In [ ]:
!pip install pydantic[email]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.8 MB/s eta 0:00:00


In [ ]:
from pydantic import BaseModel, EmailStr

class Book(BaseModel):
    title: str
    author: str
    year: int
    available: bool = True

class User(BaseModel):
    name: str
    email: EmailStr
    membership_id: str

if __name__ == "__main__":
    book1 = Book(
        title="Заповедник",
        author="Сергей Довлатов",
        year=1983,
        available=True
    )
    print("Книга создана:")
    print(book1)
    print()

    user1 = User(
        name="Егор Степанов",
        email="egorstepanov@example.com",
        membership_id="LIB-2025-001"
    )
    print("Пользователь создан:")
    print(user1)
    print()

    try:
        invalid_user = User(
            name="Егор Нестепанов",
            email="invalid-email",
            membership_id="LIB-2025-002"
        )
    except Exception as e:
        print(f"Ошибка валидации: {type(e).__name__}")
        print(f"Детали: {e}")

Книга создана:
title='Заповедник' author='Сергей Довлатов' year=1983 available=True

Пользователь создан:
name='Егор Степанов' email='egorstepanov@example.com' membership_id='LIB-2025-001'

Ошибка валидации: ValidationError
Детали: 1 validation error for User
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='invalid-email', input_type=str]


## Задача 2. Функции с аннотациями типов

Напишите следующие функции, используя аннотации типов:

1. `add_book(...) -> ...`
2. `find_book(...) -> ...`
3. `is_book_borrow(...) -> ...`
4. `return_book(...) -> ...`

In [ ]:
from pydantic import BaseModel, EmailStr
from typing import Optional

class Book(BaseModel):
    title: str
    author: str
    year: int
    available: bool = True

class User(BaseModel):
    name: str
    email: EmailStr
    membership_id: str

library: list[Book] = []

def add_book(book: Book) -> None:
    library.append(book)

def find_book(title: str) -> Optional[Book]:
    for book in library:
        if book.title == title:
            return book
    return None

def borrow_book(title: str) -> bool:
    book = find_book(title)
    if book and book.available:
        book.available = False
        return True
    return False

def return_book(title: str) -> bool:
    book = find_book(title)
    if book and not book.available:
        book.available = True
        return True
    return False

if __name__ == "__main__":
    book1 = Book(title="Заповедник", author="Сергей Довлатов", year=1983)
    book2 = Book(title="Преступление и наказание", author="Фёдор Достоевский", year=1866)

    add_book(book1)
    add_book(book2)

    found = find_book("Заповедник")
    print(f"Найдена книга: {found}")

    success = borrow_book("Заповедник")
    print(f"Книга выдана: {success}")
    print(f"Доступность книги: {book1.available}")

    success = return_book("Заповедник")
    print(f"Книга возвращена: {success}")
    print(f"Доступность книги: {book1.available}")

Найдена книга: title='Заповедник' author='Сергей Довлатов' year=1983 available=True
Книга выдана: True
Доступность книги: False
Книга возвращена: True
Доступность книги: True


## Задача 3. Расширенная модель и валидация

1. Создайте модель `Library`:
    - `books`: …
    - `users`: …
2. Добавьте в модель `Book` поле `categories: List[str]` с валидацией.
3. Реализуйте метод `total_books() -> ...` для модели `Library`.

In [ ]:
from pydantic import BaseModel, EmailStr, field_validator
from typing import Optional

class Book(BaseModel):
    title: str
    author: str
    year: int
    available: bool = True
    categories: list[str] = []

    @field_validator('categories')
    @classmethod
    def validate_categories(cls, v):
        if not v:
            raise ValueError('Список категорий не может быть пустым')
        if len(v) != len(set(v)):
            raise ValueError('Категории должны быть уникальными')
        return v

class User(BaseModel):
    name: str
    email: EmailStr
    membership_id: str

class Library(BaseModel):
    books: list[Book] = []
    users: list[User] = []

    def total_books(self) -> int:
        return len(self.books)

def add_book(library: Library, book: Book) -> None:
    library.books.append(book)

def find_book(library: Library, title: str) -> Optional[Book]:
    for book in library.books:
        if book.title == title:
            return book
    return None

def borrow_book(library: Library, title: str) -> bool:
    book = find_book(library, title)
    if book and book.available:
        book.available = False
        return True
    return False

def return_book(library: Library, title: str) -> bool:
    book = find_book(library, title)
    if book and not book.available:
        book.available = True
        return True
    return False

if __name__ == "__main__":
    library = Library()

    book1 = Book(
        title="Заповедник",
        author="Сергей Довлатов",
        year=1983,
        categories=["Современная проза", "Повесть"]
    )
    book2 = Book(
        title="Преступление и наказание",
        author="Фёдор Достоевский",
        year=1866,
        categories=["Классика", "Психология"]
    )

    add_book(library, book1)
    add_book(library, book2)

    user1 = User(
        name="Егор Степанов",
        email="egorstepanov@example.com",
        membership_id="LIB-2025-001"
    )
    library.users.append(user1)

    print(f"Всего книг в библиотеке: {library.total_books()}")

    found = find_book(library, "Заповедник")
    print(f"Найдена книга: {found.title if found else 'Не найдена'}")

    success = borrow_book(library, "Заповедник")
    print(f"Книга выдана: {success}")

    success = return_book(library, "Заповедник")
    print(f"Книга возвращена: {success}")

    try:
        invalid_book = Book(
            title="Тест",
            author="Автор",
            year=2000,
            categories=["Категория", "Категория"]
        )
    except Exception as e:
        print(f"Ошибка валидации: {e}")

Всего книг в библиотеке: 2
Найдена книга: Заповедник
Книга выдана: True
Книга возвращена: True
Ошибка валидации: 1 validation error for Book
categories
  Value error, Категории должны быть уникальными [type=value_error, input_value=['Категория', 'Категория'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error


## Задача 4. Обработка ошибок и исключения

1. Создайте исключение `BookNotAvailable`.
2. Измените функцию `is_book_borrow`, чтобы она вызывала `BookNotAvailable` при необходимости.
3. Напишите декоратор `log_operation` для логирования операций с книгами.

In [ ]:
from pydantic import BaseModel, EmailStr, field_validator
from typing import Optional
from functools import wraps
from datetime import datetime

class BookNotAvailableError(Exception):
    pass

class Book(BaseModel):
    title: str
    author: str
    year: int
    available: bool = True
    categories: list[str] = []

    @field_validator('categories')
    @classmethod
    def validate_categories(cls, v):
        if not v:
            raise ValueError('Список категорий не может быть пустым')
        if len(v) != len(set(v)):
            raise ValueError('Категории должны быть уникальными')
        return v

class User(BaseModel):
    name: str
    email: EmailStr
    membership_id: str

class Library(BaseModel):
    books: list[Book] = []
    users: list[User] = []

    def total_books(self) -> int:
        return len(self.books)

def log_operation(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"[{timestamp}] Вызов функции: {func.__name__}")
        try:
            result = func(*args, **kwargs)
            print(f"[{timestamp}] Функция {func.__name__} выполнена успешно")
            return result
        except Exception as e:
            print(f"[{timestamp}] Ошибка в функции {func.__name__}: {e}")
            raise
    return wrapper

@log_operation
def add_book(library: Library, book: Book) -> None:
    library.books.append(book)

@log_operation
def find_book(library: Library, title: str) -> Optional[Book]:
    for book in library.books:
        if book.title == title:
            return book
    return None

@log_operation
def borrow_book(library: Library, title: str) -> bool:
    book = find_book(library, title)
    if not book:
        raise BookNotAvailableError(f"Книга '{title}' не найдена в библиотеке")
    if not book.available:
        raise BookNotAvailableError(f"Книга '{title}' уже выдана")
    book.available = False
    return True

@log_operation
def return_book(library: Library, title: str) -> bool:
    book = find_book(library, title)
    if not book:
        raise BookNotAvailableError(f"Книга '{title}' не найдена в библиотеке")
    if book.available:
        raise BookNotAvailableError(f"Книга '{title}' не была выдана")
    book.available = True
    return True

if __name__ == "__main__":
    library = Library()

    book1 = Book(
        title="Заповедник",
        author="Сергей Довлатов",
        year=1983,
        categories=["Современная проза", "Повесть"]
    )
    book2 = Book(
        title="Преступление и наказание",
        author="Фёдор Достоевский",
        year=1866,
        categories=["Классика", "Психология"]
    )

    add_book(library, book1)
    add_book(library, book2)

    print(f"\nВсего книг в библиотеке: {library.total_books()}\n")

    try:
        borrow_book(library, "Заповедник")
        print("Книга 'Заповедник' успешно выдана\n")
    except BookNotAvailableError as e:
        print(f"Ошибка: {e}\n")

    try:
        borrow_book(library, "Заповедник")
    except BookNotAvailableError as e:
        print(f"Ошибка: {e}\n")

    try:
        return_book(library, "Заповедник")
        print("Книга 'Заповедник' успешно возвращена\n")
    except BookNotAvailableError as e:
        print(f"Ошибка: {e}\n")

    try:
        borrow_book(library, "Несуществующая книга")
    except BookNotAvailableError as e:
        print(f"Ошибка: {e}\n")

[2025-10-25 11:32:08] Вызов функции: add_book
[2025-10-25 11:32:08] Функция add_book выполнена успешно
[2025-10-25 11:32:08] Вызов функции: add_book
[2025-10-25 11:32:08] Функция add_book выполнена успешно

Всего книг в библиотеке: 2

[2025-10-25 11:32:08] Вызов функции: borrow_book
[2025-10-25 11:32:08] Вызов функции: find_book
[2025-10-25 11:32:08] Функция find_book выполнена успешно
[2025-10-25 11:32:08] Функция borrow_book выполнена успешно
Книга 'Заповедник' успешно выдана

[2025-10-25 11:32:08] Вызов функции: borrow_book
[2025-10-25 11:32:08] Вызов функции: find_book
[2025-10-25 11:32:08] Функция find_book выполнена успешно
[2025-10-25 11:32:08] Ошибка в функции borrow_book: Книга 'Заповедник' уже выдана
Ошибка: Книга 'Заповедник' уже выдана

[2025-10-25 11:32:08] Вызов функции: return_book
[2025-10-25 11:32:08] Вызов функции: find_book
[2025-10-25 11:32:08] Функция find_book выполнена успешно
[2025-10-25 11:32:08] Функция return_book выполнена успешно
Книга 'Заповедник' успешно 